# TP — Détection de beaconing C2

## Objectif

On dispose d'un dataframe contenant les logs de connexions sortantes sur un réseau d'entreprise.

Certains hôtes sont compromis et contactent régulièrement un serveur « Commande et contrôle » (C2). Le but du TD est de :
- identifier les hôtes compromis ;
- extraire les informations de compromission (IOC).

**Consignes :** pour chaque encadré « ✏️ Exercice », complétez la fonction
demandée puis exécutez la cellule de vérification qui la suit, ou rédigez la
réponse à la question posée (exercices sans code).

## Préambule : chargement des données

Le fichier `logs_reseau.csv` contient les connexions sortantes observées pendant
12 h (08:00 → 20:00) sur un réseau d'entreprise.

| colonne | description |
| --- | --- |
| `timestamp` | instant de la connexion |
| `src_ip` | hôte source (interne) |
| `dst_ip` | IP de destination |
| `domain` | domaine contacté (nom DNS) |
| `dst_port` | port de destination |
| `bytes_out` | octets envoyés |

In [ ]:
import statistics
from datetime import datetime, timedelta

import pandas as pd

pd.set_option("display.max_rows", 100)  # afficher les tableaux en entier

df_logs_reseau = pd.read_csv("logs_reseau.csv", parse_dates=["timestamp"])
print(df_logs_reseau.shape)
df_logs_reseau.head()

In [ ]:
df_logs_reseau.info()

## Préambule : quelques outils pandas

Le TD utilise, en plus de `groupby` / `agg`, les méthodes suivantes. On les
illustre ici sur un petit dataframe `mini`.

| méthode | rôle |
| --- | --- |
| `df.sort_values("col")` | trier les lignes selon une colonne (`ascending=False` : décroissant) |
| `serie.nunique()` | nombre de valeurs **distinctes** (≠ `count()`, nombre de valeurs) |
| `serie.isin(ensemble)` | booléen par ligne : la valeur est-elle dans l'ensemble ? |
| `~masque` | négation d'un masque booléen (« **pas** dans l'ensemble ») |
| `serie.idxmax()` / `serie.idxmin()` | **étiquette d'index** de la plus grande / plus petite valeur |
| `df.drop_duplicates("col")` | garder une seule ligne par valeur de `col` |

In [ ]:
mini = pd.DataFrame({
    "src_ip": ["10.0.0.1", "10.0.0.1", "10.0.0.2", "10.0.0.2", "10.0.0.2", "10.0.0.3"],
    "domain": ["google.com", "ntp.org", "google.com", "google.com", "evil.xyz", "google.com"],
    "bytes_out": [1200, 48, 800, 3000, 512, 450],
})
mini

Trier par taille envoyée, de la plus grande à la plus petite :

In [ ]:
mini.sort_values("bytes_out", ascending=False)

Par domaine : nombre de connexions (`count`) et nombre d'hôtes **distincts**
(`nunique`). `google.com` reçoit 4 connexions, mais de 3 hôtes seulement.

In [ ]:
print(mini.groupby("domain")["src_ip"].count())
print()
print(mini.groupby("domain")["src_ip"].nunique())

Filtrer par appartenance : `isin` garde les lignes dont le domaine est dans
l'ensemble, `~` inverse le masque (on **retire** ces domaines).

In [ ]:
connus = {"google.com", "ntp.org"}
print(mini["domain"].isin(connus).tolist())
mini[~mini["domain"].isin(connus)]

`max()` donne la plus grande **valeur**, `idxmax()` l'**étiquette d'index** où
elle se trouve. Sur le résultat d'un `groupby`, l'index est la clé de groupe :
`idxmax()` répond donc à « quel hôte ? ».

In [ ]:
volume = mini.groupby("src_ip")["bytes_out"].sum()
print(volume)
print("max :", volume.max(), "| idxmax :", volume.idxmax(), "| idxmin :", volume.idxmin())

Une seule ligne par domaine (la première rencontrée) :

In [ ]:
mini.drop_duplicates("domain")

## 1. Beaconing « classique »

Un implant « beacon » rappelle son serveur C2 à intervalle **régulier**. Objectif :
repérer le couple (hôte, domaine) au rythme le plus régulier.

Pour cela, on calcule le « coefficient de variation » (CV) des intervalles de temps entre requêtes successives :

   - `CV ≈ 0` → intervalle quasi constant (beacon *ou* tâche planifiée) ;
   - `CV` élevé → navigation humaine, irrégulière.

Étant donnée une liste **triée** de timestamps de requêtes, le coefficient de variation se calcule ainsi :
- on construit la liste des intervalles de temps, **en secondes**, entre chaque couple de requêtes successives ;
- sur cette liste, `CV = écart-type / moyenne`.

**Méthode, pas à pas :**

1. **Regrouper en listes** : fabriquer un DataFrame où chaque ligne est un couple
   `(src_ip, domain)` et où l'on range ses timestamps dans une **liste** (triée).
2. **Transformer chaque liste en écarts** : une petite fonction Python
   (`get_diffs`) qui renvoie les temps inter-arrivées, en secondes.
3. **Résumer en CV** : calculer le coefficient de variation pour chacune de ces listes.

### 1.1 Calculer un CV à la main

L'écart-type utilisé dans ce TD est l'écart-type **d'échantillon** (celui de
`statistics.stdev` et de `pandas` `.std()`). Pour $n$ écarts $x_1, \dots, x_n$
de moyenne $m$ :

$$s = \sqrt{\frac{(x_1 - m)^2 + \dots + (x_n - m)^2}{n - 1}} \qquad CV = \frac{s}{m}$$

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 1</legend>

Trois couples (hôte, domaine) ont été observés sur une courte fenêtre :

| couple | instants des connexions |
| --- | --- |
| A : `10.0.0.21 → cdn-sync.example.net` | 08:00:00, 08:01:00, 08:02:00, 08:03:00 |
| B : `10.0.0.22 → telemetry.example.net` | 08:00:00, 08:00:55, 08:01:55, 08:03:00 |
| C : `10.0.0.23 → news.example.org` | 08:00:00, 08:01:40, 08:11:40, 08:30:00 |

1. Pour chaque couple, écrivez la liste des écarts en secondes entre connexions successives.
2. Calculez la moyenne, l'écart-type et le CV de chaque liste.
3. Classez les trois couples du plus régulier au moins régulier. Lequel ressemble à de la navigation humaine ?
4. Peut-on conclure que A est un beacon malveillant ? Que dire de B ?

</fieldset>

*Votre réponse :*

<!-- TOCLEAR -->
<fieldset style="border: 3px solid #2e7d32; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #2e7d32; font-weight: bold; font-size: 1.5em; padding: 0 6px">💡 Réponse</legend>

1. A : `60, 60, 60` ; B : `55, 60, 65` ; C : `100, 600, 1100`.
2. - A : $m = 60$, tous les écarts à la moyenne sont nuls → $s = 0$, **CV = 0**.
   - B : $m = 60$, $s = \sqrt{(25 + 0 + 25)/2} = 5$ → **CV = 5/60 ≈ 0,08**.
   - C : $m = 600$, $s = \sqrt{(250\,000 + 0 + 250\,000)/2} = 500$ → **CV = 500/600 ≈ 0,83**.
3. A (0) < B (0,08) < C (0,83). C, irrégulier, ressemble à de la navigation humaine.
4. Non : un CV nul dit seulement que le trafic est **périodique**. Une mise à
   jour, une synchronisation ou NTP sont aussi périodiques ; il faut croiser
   avec d'autres indices (domaine connu ou non, taille des messages…). B est
   presque aussi régulier que A : un léger aléa (*jitter*) de quelques secondes
   autour de 60 s ne suffit pas à masquer le rythme.

</fieldset>

On vérifie ici les CV avec le module `statistics` :

In [ ]:
for nom, ecarts in [("A", [60, 60, 60]), ("B", [55, 60, 65]), ("C", [100, 600, 1100])]:
    m, s = statistics.mean(ecarts), statistics.stdev(ecarts)
    print(f"{nom} : moyenne = {m:6.1f}  écart-type = {s:6.1f}  CV = {s / m:.2f}")

### 1.2 Regrouper les timestamps par couple (src_ip, domain)

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 2</legend>

Complétez `timestamps_par_src_dest` : elle renvoie un dataframe indexé par `(src_ip, domain)`, avec une colonne `timestamps` contenant la liste triée des timestamps de connexion de chaque couple.

</fieldset>

In [ ]:
def timestamps_par_src_dest(df: pd.DataFrame) -> pd.DataFrame:
    """
    À partir du dataframe `logs_reseau`, retourner un dataframe qui a en index
    `(src_ip, domain)`, et une colonne `timestamps`, qui contient une liste triée
    de tous les timestamps de connexion associés à `(src_ip, domain)`
    """
    # BEGIN
    # raise NotImplementedError
    return (
        df.sort_values("timestamp")
        .groupby(["src_ip", "domain"])
        .agg(timestamps=("timestamp", list))
    )
    # END

In [ ]:
ts_per_src_dest = timestamps_par_src_dest(df_logs_reseau)
nb_couples = df_logs_reseau[["src_ip", "domain"]].drop_duplicates().shape[0]
assert ts_per_src_dest.shape[0] == nb_couples
assert "timestamps" in ts_per_src_dest.columns
assert ts_per_src_dest["timestamps"].map(lambda ts: ts == sorted(ts)).all()
print("ex 2 OK :", nb_couples, "couples (src_ip, domain) distincts")

### 1.3 Calculer les écarts entre requêtes

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 3</legend>

Complétez `get_diffs` : elle renvoie, dans l'ordre chronologique, les écarts en secondes entre instants successifs d'une liste de timestamps (liste vide s'il y a moins de deux instants).

</fieldset>

In [ ]:
def get_diffs(timestamps: list[datetime]) -> list[float]:
    """Temps (en secondes) entre instants successifs, dans l'ordre chronologique.

    La liste est triée au besoin : aucune hypothèse sur l'ordre en entrée.
    """
    # BEGIN
    # raise NotImplementedError
    tri = sorted(timestamps)
    res = []
    for i in range(len(tri) - 1):
        res.append((tri[i + 1] - tri[i]).total_seconds())
    return res
    # END

In [ ]:
t0 = datetime(2026, 10, 14, 8, 0, 0)
assert get_diffs([t0, t0 + timedelta(seconds=60), t0 + timedelta(seconds=120)]) == [60.0, 60.0]
assert get_diffs([t0 + timedelta(seconds=120), t0, t0 + timedelta(seconds=60)]) == [60.0, 60.0]
assert get_diffs([t0]) == []
assert get_diffs([]) == []
print("ex 3 OK")

### 1.4 Résumer en coefficient de variation

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 4</legend>

Complétez `cv_destinations` : elle calcule, à partir du dataframe de `timestamps_par_src_dest`, le nombre `n` d'écarts et le coefficient de variation `cv` de chaque couple `(src_ip, domain)`, en ignorant les couples ayant moins de `n_min` écarts.

</fieldset>

In [ ]:
def cv_destinations(df: pd.DataFrame, n_min: int = 20) -> pd.DataFrame:
    """Coefficient de variation des temps inter-arrivées par couple (src_ip, domain).

    Colonnes retournées : `src_ip`, `domain`, `n`, `cv`. On ignore les couples trop
    rares (moins de `n_min` écarts) : un CV sur 2 points ne veut rien dire.
    """
    # BEGIN
    # raise NotImplementedError
    ## Indice : le module `statistics` fournit `statistics.mean(liste)` et
    ## `statistics.stdev(liste)` : moyenne et écart-type d'une liste de nombres.
    df = df.copy()
    df["diffs"] = df["timestamps"].map(get_diffs)
    df["n"] = df["diffs"].map(len)
    df = df[df["n"] >= n_min]

    df["moyenne"] = df["diffs"].map(statistics.mean)
    df["ecart_type"] = df["diffs"].map(statistics.stdev)
    df["cv"] = df["ecart_type"] / df["moyenne"]
    return df.reset_index()[["src_ip", "domain", "n", "cv"]]
    # END

In [ ]:
def check_cv_destinations():
    # données à la main : écarts réguliers de 60 s -> CV = 0 ;
    # écarts (10, 30, 20) -> moyenne 20, écart-type 10, CV = 0.5 ;
    # un couple avec un seul écart est écarté (n_min = 3).
    idx = pd.MultiIndex.from_tuples(
        [("a", "x"), ("a", "y"), ("b", "z")], names=["src_ip", "domain"]
    )
    t0 = datetime(2026, 1, 1, 8, 0, 0)
    sec = lambda *s: [t0 + timedelta(seconds=v) for v in s]
    petit = pd.DataFrame(
        {"timestamps": [sec(0, 60, 120, 180, 240), sec(0, 10, 40, 60), sec(0, 5)]},
        index=idx,
    )
    res = cv_destinations(petit, n_min=3)
    assert list(res.columns) == ["src_ip", "domain", "n", "cv"], "colonnes incorrectes"
    res = res.set_index(["src_ip", "domain"])
    assert len(res) == 2, "le couple trop rare doit être écarté"
    assert res.loc[("a", "x"), "n"] == 4
    assert res.loc[("a", "x"), "cv"] == 0.0
    assert res.loc[("a", "y"), "n"] == 3
    assert res.loc[("a", "y"), "cv"] == 0.5
    print("ex 4 OK")

check_cv_destinations()

On affiche ici tous les couples `(src_ip, domain)` retenus, triés par CV croissant.

In [ ]:
cv_destinations(timestamps_par_src_dest(df_logs_reseau)).sort_values("cv")

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 5</legend>

On lit le tableau ci-dessus.

1. Quels domaines occupent les premières lignes ? Ces connexions sont-elles malveillantes ?
2. Repérez, dans le haut du tableau, un couple au CV faible dont le domaine n'est pas un service connu.
   Son CV (≈ 0,01) est plus élevé que celui des mises à jour (≈ 0,003), alors que
   le jitter est du même ordre (quelques secondes) des deux côtés : pourquoi ?
3. Les domaines `free-prize-claim.top` et `x9f2k7q4m1z.xyz` ont un nom inquiétant.
   Quel est leur CV ? Qu'en conclure ?
4. `ntp.org` est contacté toutes les heures par plusieurs hôtes, mais n'apparaît
   pas dans le tableau. Pourquoi ? Que vaudrait le CV d'un couple n'ayant que
   3 connexions, espacées de 300 s puis 310 s ? Quel est le rôle de `n_min` ?

</fieldset>

*Votre réponse :*

<!-- TOCLEAR -->
<fieldset style="border: 3px solid #2e7d32; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #2e7d32; font-weight: bold; font-size: 1.5em; padding: 0 6px">💡 Réponse</legend>

1. Des mises à jour et services de synchronisation (`windowsupdate.com`,
   `av-updates.example.net`, `metrics-updates.example.net`,
   `sync-backup.example.net`) : périodiques mais légitimes. Ce sont des
   **faux positifs** du critère « CV faible ».
2. `10.0.0.66 → sync-updates.example.net` (nom proche de `sync-backup`, mais
   inconnu). Le CV divise l'écart-type par la moyenne : pour un même jitter
   (±5 s), un intervalle court (420 s) donne un CV plus grand qu'un
   intervalle long (1200–1800 s). Il reste très loin des CV humains (≥ 0,7).
3. Leur CV est d'environ 0,8–0,9, comme la navigation humaine : leur rythme
   n'a rien d'un beacon. Le nom seul ne prouve rien (et dans l'autre sens, un
   C2 peut porter un nom anodin comme `sync-updates.example.net`).
4. Chaque hôte ne contacte `ntp.org` que 12 fois en 12 h, soit 11 écarts
   < `n_min = 20` : le couple est écarté. Avec 3 connexions (2 écarts 300 et
   310), $m = 305$, $s ≈ 7$, CV ≈ 0,02 : on dirait un beacon, par pur hasard.
   Avec 2 connexions (1 seul écart), l'écart-type n'est même pas défini.
   `n_min` garantit assez de points pour que le CV soit significatif.

</fieldset>

Le module 7 du cours calcule ces mêmes statistiques en une seule chaîne
pandas : `groupby(...).diff()` donne, pour chaque ligne, l'écart avec la ligne
précédente **du même groupe** (`NaT` pour la première connexion de chaque
couple). La cellule suivante vérifie que l'on retrouve les CV de `cv_destinations`.

In [ ]:
df_tri = df_logs_reseau.sort_values("timestamp")
df_tri["dt"] = df_tri.groupby(["src_ip", "domain"])["timestamp"].diff().dt.total_seconds()

stats = (df_tri.dropna(subset=["dt"])
               .groupby(["src_ip", "domain"])["dt"]
               .agg(n="count", moyenne="mean", ecart_type="std"))
stats["cv"] = stats["ecart_type"] / stats["moyenne"]
stats = stats[stats["n"] >= 20]

reference = cv_destinations(timestamps_par_src_dest(df_logs_reseau)).set_index(["src_ip", "domain"])
assert set(stats.index) == set(reference.index)
assert ((stats["cv"] - reference["cv"]).abs() < 1e-9).all()
print("mêmes CV avec groupby().diff() :", len(stats), "couples")

### 1.5 Liste blanche : les faux positifs périodiques

Le tri par CV ci-dessus montre que les couples les plus réguliers ne sont pas
forcément malveillants : les mises à jour (Windows, antivirus, NTP…) contactent
elles aussi leur serveur à intervalle quasi constant — avec un petit jitter réseau
du même ordre que celui du beacon. Ce sont des **faux positifs**.

L'équipe SOC fournit la liste des domaines légitimes connus (liste blanche) :
on les écarte avant de chercher le beacon. Cette liste est **réutilisée dans la
suite du TD** chaque fois qu'il faut écarter le trafic légitime.

Domaines whitelistés : `windowsupdate.com`, `ntp.org`, `av-updates.example.net`,
`sync-backup.example.net`, `metrics-updates.example.net`.

In [ ]:
# Domaines légitimes périodiques (liste blanche fournie par le SOC).
# À réutiliser chaque fois qu'il faut écarter le trafic légitime.
DOMAINES_WHITELIST = {
    "windowsupdate.com",
    "ntp.org",
    "av-updates.example.net",
    "sync-backup.example.net",
    "metrics-updates.example.net",
}

### 1.6 Détection du beacon

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 6</legend>

Complétez `detecter_beacon_timing` : elle renvoie le couple `(src_ip, domain)` de CV minimal, en écartant les domaines de `DOMAINES_WHITELIST` (utiliser `timestamps_par_src_dest` et `cv_destinations`).

</fieldset>

In [ ]:
def detecter_beacon_timing(df: pd.DataFrame) -> tuple[str, str]:
    """Renvoie (src_ip, domain) du beacon le plus régulier, hors liste blanche."""
    # BEGIN
    # raise NotImplementedError
    stats = cv_destinations(timestamps_par_src_dest(df))
    # Certains domaines légitimes (mises à jour, NTP…) sont *aussi* périodiques :
    # on les écarte avant de chercher le CV minimal.
    hors_whitelist = stats[~stats["domain"].isin(DOMAINES_WHITELIST)]
    suspect = hors_whitelist.loc[hors_whitelist["cv"].idxmin()]
    return suspect["src_ip"], suspect["domain"]
    # END

In [ ]:
src, domaine = detecter_beacon_timing(df_logs_reseau)
assert (src, domaine) == ("10.0.0.66", "sync-updates.example.net"), f"trouvé : {src} -> {domaine}"
print("ex 6 OK :", src, "->", domaine)

### 1.7 Extraction des IOC

Le couple (hôte, domaine) suspect étant identifié, on extrait ses **IOC**
(indicateurs de compromission) : les IP de destination distinctes
contactées par cet hôte pour ce domaine.

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 7</legend>

Complétez `extraire_ioc_beacon` : elle renvoie un dataframe des destinations distinctes (une ligne par IP de destination, colonnes `dst_ip` et `domain`) contactées par `(src_ip, domain)`, trié par `dst_ip`.

</fieldset>

In [ ]:
def extraire_ioc_beacon(df: pd.DataFrame, src_ip: str, domain: str) -> pd.DataFrame:
    """IOC du beacon : destinations distinctes contactées par `(src_ip, domain)`.

    Renvoie un dataframe des destinations distinctes (une ligne par IP de
    destination), trié par `dst_ip`.
    """
    # BEGIN
    # raise NotImplementedError
    return (
        df[(df["src_ip"] == src_ip) & (df["domain"] == domain)][["dst_ip", "domain"]]
        .drop_duplicates("dst_ip")
        .sort_values("dst_ip")
        .reset_index(drop=True)
    )
    # END

In [ ]:
src, domaine = detecter_beacon_timing(df_logs_reseau)
ioc_beacon = extraire_ioc_beacon(df_logs_reseau, src, domaine)
assert len(ioc_beacon) == 1
assert ioc_beacon.iloc[0]["dst_ip"] == "185.165.29.10"
assert ioc_beacon.iloc[0]["domain"] == "sync-updates.example.net"
print("ex 7 OK :", len(ioc_beacon), "IOC")

### 1.8 Sans liste blanche : la stabilité du payload

La liste blanche a un défaut : il faut la tenir à jour, et un attaquant peut
choisir un nom de domaine qui ressemble à un service légitime. Un second
critère ne dépend d'aucune liste : la **taille des messages** (`bytes_out`).
Un beacon envoie à chaque fois le même petit message (« je suis là, des
ordres ? ») ; un téléchargement de mise à jour, lui, varie d'une fois sur l'autre.

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 8</legend>

Complétez `detecter_payload_constant` : elle renvoie la liste triée des couples `(src_ip, domain)` ayant au moins `n_min` **connexions** et dont `bytes_out` ne prend qu'**une seule valeur**. Elle n'utilise pas `DOMAINES_WHITELIST`.

</fieldset>

In [ ]:
def detecter_payload_constant(df: pd.DataFrame, n_min: int = 20) -> list[tuple[str, str]]:
    """Couples `(src_ip, domain)` d'au moins `n_min` connexions à `bytes_out` constant."""
    # BEGIN
    # raise NotImplementedError
    ## Indice : `agg` accepte plusieurs colonnes de sortie, par ex.
    ## `.agg(n=("bytes_out", "count"), nb_tailles=("bytes_out", "nunique"))`
    stats = df.groupby(["src_ip", "domain"]).agg(
        n=("bytes_out", "count"), nb_tailles=("bytes_out", "nunique")
    )
    suspects = stats[(stats["n"] >= n_min) & (stats["nb_tailles"] == 1)]
    return sorted(suspects.index)
    # END

In [ ]:
def check_payload_constant():
    # a->x : 3 connexions de 512 o (constant) ; a->y : tailles variables ;
    # b->z : constant mais une seule connexion (écarté avec n_min = 3).
    petit = pd.DataFrame({
        "src_ip": ["a", "a", "a", "a", "a", "a", "b"],
        "domain": ["x", "x", "x", "y", "y", "y", "z"],
        "bytes_out": [512, 512, 512, 100, 200, 100, 64],
    })
    assert detecter_payload_constant(petit, n_min=3) == [("a", "x")]
    trouves = detecter_payload_constant(df_logs_reseau)
    assert trouves == [("10.0.0.66", "sync-updates.example.net")], f"trouvé : {trouves}"
    print("ex 8 OK :", trouves)

check_payload_constant()

Après la vérification, la cellule suivante affiche, pour les couples
réguliers (CV < 0,1), le nombre de tailles distinctes envoyées.

In [ ]:
regularite = cv_destinations(timestamps_par_src_dest(df_logs_reseau))
tailles = (df_logs_reseau.groupby(["src_ip", "domain"])["bytes_out"]
           .nunique().rename("nb_tailles").reset_index())
(regularite[regularite["cv"] < 0.1]
    .merge(tailles, on=["src_ip", "domain"])
    .sort_values("nb_tailles")
    .head(10))

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 9</legend>

1. Pourquoi ce critère trouve-t-il le beacon sans liste blanche ?
2. Citez un service légitime qui pourrait, lui aussi, envoyer des messages de taille constante. Que faut-il en conclure ?
3. Comment l'attaquant peut-il casser ce signal ? Quel signal lui reste-t-il alors à masquer ?

</fieldset>

*Votre réponse :*

<!-- TOCLEAR -->
<fieldset style="border: 3px solid #2e7d32; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #2e7d32; font-weight: bold; font-size: 1.5em; padding: 0 6px">💡 Réponse</legend>

1. Les mises à jour sont périodiques (CV faible) mais leur taille varie à
   chaque connexion (`nb_tailles` = nombre de connexions) ; le beacon envoie
   toujours 512 octets (`nb_tailles` = 1). Les deux critères réunis (CV faible
   **et** taille constante) isolent le beacon.
2. Un *heartbeat* de supervision, une requête NTP (48 octets), un keepalive…
   Ce critère seul produit aussi des faux positifs : on **croise** les signaux
   (régularité, taille, domaine connu ou non).
3. En ajoutant un remplissage aléatoire (*padding*) à chaque message : la
   taille varie, `nb_tailles` explose. Il lui reste à casser la régularité
   (jitter, timing aléatoire) : voir 1.9 et la partie 2.

</fieldset>

### 1.9 Jitter : jusqu'où le CV tient-il ?

Pour paraître moins mécanique, un implant ajoute un aléa (*jitter*) à son
intervalle de rappel.

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 10</legend>

Un implant rappelle son C2 toutes les 60 s, avec un jitter de ±30 % : chaque intervalle est tiré au hasard, uniformément, entre 42 s et 78 s.

1. Les écarts à la moyenne vont de 0 à 18 s. Donnez un ordre de grandeur de l'écart-type, puis du CV.
2. Comparez aux CV du tableau de la partie 1.4 (mises à jour ≈ 0,003, beacon ≈ 0,01, navigation humaine ≥ 0,7). L'implant reste-t-il détectable ? Proposez un seuil, et dites ce qu'il coûte.
3. L'implant passe à un timing **totalement aléatoire** (intervalle de loi exponentielle, CV ≈ 1). Que peut encore observer le défenseur ?

</fieldset>

*Votre réponse :*

<!-- TOCLEAR -->
<fieldset style="border: 3px solid #2e7d32; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #2e7d32; font-weight: bold; font-size: 1.5em; padding: 0 6px">💡 Réponse</legend>

1. Écart-type de l'ordre de 10 s (entre 0 et 18, un peu au-dessus du milieu),
   donc CV ≈ 10 / 60 ≈ **0,15–0,2** (valeur exacte : 0,3 / √3 ≈ 0,17).
2. Oui : 0,17 reste très en dessous des CV humains (≥ 0,7). Un seuil vers
   CV < 0,3 le détecte encore, mais laisse passer davantage de trafic
   légitime un peu régulier (plus de faux positifs à trier). Plus le jitter
   augmente, plus le CV se rapproche de celui d'un humain.
3. Le rythme ne trahit plus rien. Il reste **où** l'implant se connecte :
   une plage d'IP que personne d'autre ne contacte (partie 2), des
   destinations rares (partie 4), et le volume de connexions.

</fieldset>

On vérifie ici par simulation (200 intervalles tirés au hasard) :

In [ ]:
import random

rng = random.Random(0)
avec_jitter = [rng.uniform(42, 78) for _ in range(200)]
aleatoire = [rng.expovariate(1 / 60) for _ in range(200)]
for nom, ecarts in [("jitter ±30 %", avec_jitter), ("timing aléatoire", aleatoire)]:
    print(f"{nom:17} : CV = {statistics.stdev(ecarts) / statistics.mean(ecarts):.2f}")

## 2. Timing aléatoire : il ne reste que la plage d'IP

L'implant le plus sophistiqué **randomise totalement** son timing : plus aucun
rythme à détecter. Mais il doit toujours joindre son serveur C2, hébergé dans un
`/24` que l'entreprise ne contacte jamais.

**Méthode :**

1. Écarter le domaine du beacon déjà identifié au cas 1
   (`sync-updates.example.net`).
2. Réduire chaque IP restante à son préfixe `/24`.
3. Pour chaque préfixe, calculer **deux chiffres séparés** :
   - `nb_sources` : combien d'hôtes distincts le touchent ;
   - `volume` : combien de connexions il reçoit.
4. Un préfixe **rare** (1 seule source) **et massivement contacté** = suspect.

**Note :** cet implant ne laisse aucun rythme exploitable et ajoute un
*padding* aléatoire à ses messages : ni la régularité (cas 1), ni la stabilité
du payload (cas 1), ni le comptage de destinations (cas 3) ne le détectent.
Seule la plage d'IP le trahit.

### 2.1 Préfixe /24 d'une IP

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 11</legend>

Complétez `prefixe` : elle renvoie le préfixe `/24` d'une IPv4, c'est-à-dire ses trois premiers octets (par exemple `"185.165.29"` pour `"185.165.29.10"`).

</fieldset>

In [ ]:
def prefixe(ip: str) -> str:
    """Préfixe /24 d'une IPv4."""
    # BEGIN
    # raise NotImplementedError
    return ".".join(ip.split(".")[:3])
    # END

In [ ]:
assert prefixe("185.165.29.10") == "185.165.29"
assert prefixe("10.0.0.1") == "10.0.0"
print("ex 11 OK")

In [ ]:
# DELETE
df_logs_reseau.assign(prefix=df_logs_reseau["dst_ip"].map(prefixe))

### 2.2 Hôtes sur une plage rare et massivement contactée

Pour chaque préfixe `/24` des destinations, on compte le nombre de sources
distinctes qui le contactent, ainsi que le nombre total de requêtes qu'il reçoit.
On ne garde ensuite que les préfixes à une seule source et au volume élevé,
puis on remonte aux hôtes sources correspondants.

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 12</legend>

Complétez `hotes_plage_rare` : elle renvoie la liste triée des hôtes source qui touchent un `/24` ayant une seule source et au moins `vol_min` connexions, après avoir écarté le domaine `sync-updates.example.net`.

</fieldset>

In [ ]:
def hotes_plage_rare(df: pd.DataFrame, vol_min: int = 50) -> list[str]:
    """Hôtes source touchant un `/24` rare (1 source) ET massivement contacté."""
    # BEGIN
    # raise NotImplementedError
    ## Le beacon du cas 1 est lui aussi sur un /24 rare : on exclut
    ## son domaine, déjà identifié, pour ne garder que le nouvel implant.
    df = df[~df["domain"].isin({"sync-updates.example.net"})].copy()
    df["prefix"] = df["dst_ip"].map(prefixe)

    # deux signaux, calculés séparément :
    nb_sources = df.groupby("prefix")["src_ip"].nunique()  # combien d'hôtes distincts ?
    volume = df.groupby("prefix")["timestamp"].count()      # combien de connexions ?

    rares = nb_sources[(nb_sources == 1) & (volume >= vol_min)].index
    return sorted(df[df["prefix"].isin(rares)]["src_ip"].unique())
    # END

In [ ]:
suspects = hotes_plage_rare(df_logs_reseau)
assert suspects == ["10.0.0.68"], f"trouvé : {suspects}"
print("ex 12 OK :", suspects)

### 2.3 Extraction des IOC

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 13</legend>

Complétez `extraire_ioc_plage` : elle renvoie les destinations distinctes (une ligne par IP de destination, colonnes `dst_ip` et `domain`) contactées par l'hôte suspect, triées par `dst_ip`.

</fieldset>

In [ ]:
def extraire_ioc_plage(df: pd.DataFrame, src_ip: str) -> pd.DataFrame:
    """IOC associés à une IP source : les destinations (IP + domaine) contactées.

    Renvoie un dataframe des destinations distinctes (une ligne par IP de
    destination), trié par `dst_ip`.
    """
    # BEGIN
    # raise NotImplementedError
    return (
        df[df["src_ip"] == src_ip][["dst_ip", "domain"]]
        .drop_duplicates("dst_ip")
        .sort_values("dst_ip")
        .reset_index(drop=True)
    )
    # END

In [ ]:
suspect = hotes_plage_rare(df_logs_reseau)[0]
ioc_plage = extraire_ioc_plage(df_logs_reseau, suspect)
assert len(ioc_plage) == 5
assert ioc_plage["dst_ip"].is_unique
assert (ioc_plage["dst_ip"].str.startswith("91.240.118.")).all()
print("ex 13 OK :", len(ioc_plage), "IOC")

## 3. Fast-flux : la destination qui tourne

Certains C2 changent d'IP **et** de domaine à chaque connexion pour échapper aux
filtres : c'est le **fast-flux** (ou DGA). Chaque domaine n'apparaissant qu'une
fois, grouper par domaine est inutile.

**Méthode :** regarder l'hôte *source*. Un hôte légitime contacte un petit nombre
de destinations. Un hôte en fast-flux contacte un grand nombre d'IP **distinctes**.

### 3.1 Hôte aux destinations multiples

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 14</legend>

Complétez `detecter_fastflux` : elle renvoie l'hôte source qui contacte le plus d'IP de destination distinctes.

</fieldset>

In [ ]:
def detecter_fastflux(df: pd.DataFrame) -> str:
    """Renvoie l'hôte source qui contacte le plus d'IP de destination distinctes."""
    # BEGIN
    # raise NotImplementedError
    return df.groupby("src_ip")["dst_ip"].nunique().idxmax()
    # END

In [ ]:
hote = detecter_fastflux(df_logs_reseau)
assert hote == "10.0.0.67", f"trouvé : {hote}"
print("ex 14 OK :", hote)

### 3.2 Extraction des IOC

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 15</legend>

Complétez `extraire_ioc` : elle renvoie les destinations distinctes (une ligne par IP de destination, colonnes `dst_ip` et `domain`) contactées par l'hôte `src_ip`, triées par `dst_ip`.

</fieldset>

In [ ]:
def extraire_ioc(df: pd.DataFrame, src_ip: str) -> pd.DataFrame:
    """IOC associés à une IP source : les destinations (IP + domaine) contactées.

    Renvoie un dataframe des destinations distinctes (une ligne par IP de
    destination), trié par `dst_ip`.
    """
    # BEGIN
    # raise NotImplementedError
    return (
        df[df["src_ip"] == src_ip][["dst_ip", "domain"]]
        .drop_duplicates("dst_ip")
        .sort_values("dst_ip")
        .reset_index(drop=True)
    )
    # END

In [ ]:
hote = detecter_fastflux(df_logs_reseau)
ioc = extraire_ioc(df_logs_reseau, hote)
assert len(ioc) == 40
assert ioc["dst_ip"].is_unique
print("ex 15 OK :", len(ioc), "IOC (IP de destination distinctes)")

## 4. Quand le comptage ne suffit plus

Le réseau grossit : on observe désormais une centaine d'hôtes, qui contactent
chacun des centaines d'IP de destination distinctes (CDN, edge, caches…).

Un hôte compromis renouvelle ses destinations à chaque connexion.

### 4.1 Chargement du grand réseau

In [ ]:
df_logs_reseau_grand = pd.read_csv("logs_reseau_grand.csv", parse_dates=["timestamp"])
print(df_logs_reseau_grand.shape)
df_logs_reseau_grand.head()

### 4.2 Détection de l'hôte compromis

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 16</legend>

Complétez `detecter_hote_compromis` : elle renvoie l'hôte compromis du grand réseau.

</fieldset>

In [ ]:
def detecter_hote_compromis(df: pd.DataFrame) -> str:
    """Renvoie l'hôte compromis du grand réseau."""
    # BEGIN
    # raise NotImplementedError
    nb_sources = df.groupby("dst_ip")["src_ip"].nunique()
    ip_rares = nb_sources[nb_sources == 1].index
    rare_par_hote = df[df["dst_ip"].isin(ip_rares)].groupby("src_ip")["dst_ip"].nunique()
    return rare_par_hote.idxmax()
    # END

In [ ]:
victime = detecter_hote_compromis(df_logs_reseau_grand)
assert victime == "10.0.0.167", f"trouvé : {victime}"
print("ex 16 OK :", victime)

### 4.3 Lire la chaîne des destinations rares

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 17</legend>

On considère la chaîne suivante, appliquée au petit dataframe `petit` ci-dessous :

```python
nb_sources = df.groupby("dst_ip")["src_ip"].nunique()
ip_rares = nb_sources[nb_sources == 1].index
rare_par_hote = df[df["dst_ip"].isin(ip_rares)].groupby("src_ip")["dst_ip"].nunique()
resultat = rare_par_hote.idxmax()
```

| | src_ip | dst_ip |
| --- | --- | --- |
| 0 | 10.0.0.1 | 198.18.0.1 |
| 1 | 10.0.0.2 | 198.18.0.1 |
| 2 | 10.0.0.3 | 198.18.0.1 |
| 3 | 10.0.0.1 | 198.18.0.2 |
| 4 | 10.0.0.2 | 198.18.0.2 |
| 5 | 10.0.0.3 | 193.29.0.7 |
| 6 | 10.0.0.3 | 193.29.0.8 |
| 7 | 10.0.0.2 | 185.220.3.4 |
| 8 | 10.0.0.3 | 193.29.0.7 |

1. Décrivez en français ce que contiennent `nb_sources`, `ip_rares` et `rare_par_hote` (que représentent l'index et les valeurs ?).
2. Calculez à la main leur valeur sur `petit`, ainsi que `resultat`.
3. À la dernière ligne, que changerait `count()` à la place de `nunique()` ?

</fieldset>

*Votre réponse :*

<!-- TOCLEAR -->
<fieldset style="border: 3px solid #2e7d32; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #2e7d32; font-weight: bold; font-size: 1.5em; padding: 0 6px">💡 Réponse</legend>

1. - `nb_sources` : pour chaque IP de destination (index), le nombre d'hôtes
     sources **distincts** qui la contactent.
   - `ip_rares` : la liste des IP de destination contactées par **un seul** hôte.
   - `rare_par_hote` : pour chaque hôte source (index), le nombre d'IP rares
     distinctes qu'il contacte. `resultat` est l'hôte qui en contacte le plus.
2. - `nb_sources` : `198.18.0.1` → 3, `198.18.0.2` → 2, `193.29.0.7` → 1,
     `193.29.0.8` → 1, `185.220.3.4` → 1.
   - `ip_rares` : `193.29.0.7`, `193.29.0.8`, `185.220.3.4`.
   - `rare_par_hote` : `10.0.0.2` → 1, `10.0.0.3` → 2 (`10.0.0.1` n'apparaît
     pas : il ne touche aucune IP rare).
   - `resultat` : `"10.0.0.3"`.
3. `count()` compte les **connexions** : `10.0.0.3` contacte deux fois
   `193.29.0.7`, il aurait 3 au lieu de 2. Avec `nunique()`, on compte des
   **destinations différentes** : un hôte qui contacte souvent la même IP rare
   ne pèse pas plus qu'un hôte qui la contacte une fois.

</fieldset>

On vérifie ici en exécutant la chaîne sur `petit` :

In [ ]:
petit = pd.DataFrame({
    "src_ip": ["10.0.0.1", "10.0.0.2", "10.0.0.3", "10.0.0.1", "10.0.0.2",
               "10.0.0.3", "10.0.0.3", "10.0.0.2", "10.0.0.3"],
    "dst_ip": ["198.18.0.1", "198.18.0.1", "198.18.0.1", "198.18.0.2", "198.18.0.2",
               "193.29.0.7", "193.29.0.8", "185.220.3.4", "193.29.0.7"],
})
df = petit
nb_sources = df.groupby("dst_ip")["src_ip"].nunique()
ip_rares = nb_sources[nb_sources == 1].index
rare_par_hote = df[df["dst_ip"].isin(ip_rares)].groupby("src_ip")["dst_ip"].nunique()
print(nb_sources, end="\n\n")
print(list(ip_rares), end="\n\n")
print(rare_par_hote, end="\n\n")
print("resultat :", rare_par_hote.idxmax())

### 4.4 Extraction des IOC

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 18</legend>

Complétez `extraire_ioc_grand` : elle renvoie les destinations distinctes (une ligne par IP de destination, colonnes `dst_ip` et `domain`) contactées par l'hôte compromis, triées par `dst_ip`.

</fieldset>

In [ ]:
def extraire_ioc_grand(df: pd.DataFrame, src_ip: str) -> pd.DataFrame:
    """IOC associés à une IP source : les destinations (IP + domaine) contactées.

    Renvoie un dataframe des destinations distinctes (une ligne par IP de
    destination), trié par `dst_ip`.
    """
    # BEGIN
    # raise NotImplementedError
    return (
        df[df["src_ip"] == src_ip][["dst_ip", "domain"]]
        .drop_duplicates("dst_ip")
        .sort_values("dst_ip")
        .reset_index(drop=True)
    )
    # END

In [ ]:
victime = detecter_hote_compromis(df_logs_reseau_grand)
ioc_grand = extraire_ioc_grand(df_logs_reseau_grand, victime)
assert len(ioc_grand) == 400
assert ioc_grand["dst_ip"].is_unique
print("ex 18 OK :", len(ioc_grand), "IOC")

## Bilan

| cas d'étude | signal | notion pandas | hôte trouvé |
| --- | --- | --- | --- |
| 1 | régularité temporelle | `groupby` + `agg(list)` + CV | `10.0.0.66` |
| 1 | stabilité du payload | `groupby().agg(nunique)` | `10.0.0.66` |
| 2 | plage d'IP rare et concentrée | `map` + deux `groupby` | `10.0.0.68` |
| 3 | multiplicité des destinations | `groupby().nunique()` | `10.0.0.67` |
| 4 | destinations rares (une seule source) | `nunique` + `isin` | `10.0.0.167` |

**À retenir :** la détection de C2 est **heuristique** — on croise plusieurs
signaux (rythme, destinations, plage d'IP, rareté des destinations). Aucun
signal seul ne suffit, et chaque contre-mesure de l'attaquant (jitter,
padding aléatoire, fast-flux, timing aléatoire, destinations partagées) casse
un signal… mais en laisse un autre.